In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [3]:
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)

In [4]:
df_match = pd.read_csv(r'F:\ALMACENAMIENTO\DATA ANALITYCS\PORTAFOLIO\AI_Assistant_Hotels\SQL_ENRICHMENT\data_interaction__handling_hotel_match_features.csv', encoding='latin-1',dtype={'resolved': str})

In [5]:
df_full = pd.read_csv(r'F:\ALMACENAMIENTO\DATA ANALITYCS\PORTAFOLIO\AI_Assistant_Hotels\SQL_ENRICHMENT\data_interaction_hotel_full_features.csv',encoding='latin-1')

In [6]:
print("="*50)
print("DATASETS LOADED")
print("="*50)
print(f"✅ df_full (volume):      {df_full.shape[0]} rows, {df_full.shape[1]} columns")
print(f"✅ df_match (performance): {df_match.shape[0]} rows, {df_match.shape[1]} columns")
print("="*50)

DATASETS LOADED
✅ df_full (volume):      4740 rows, 20 columns
✅ df_match (performance): 3292 rows, 24 columns


In [7]:
# Ver columnas de cada dataset
print("df_full columns:")
print(df_full.columns.tolist())

print("\ndf_match columns:")
print(df_match.columns.tolist())

df_full columns:
['interaction_id', 'hotel_id', 'channel', 'language', 'request_type', 'complexity', 'hotel_name', 'city', 'country', 'hotel_type', 'rooms', 'hotel_size_category', 'timestamp', 'day_of_week', 'hour_of_day', 'day_of_week_number', 'time_period', 'month_number', 'month_name', 'quarter']

df_match columns:
['interaction_id', 'hotel_id', 'timestamp', 'channel', 'language', 'request_type', 'complexity', 'hotel_name', 'city', 'country', 'hotel_type', 'rooms', 'hotel_size_category', 'handled_by', 'response_time_s', 'response_speed', 'hour_of_day', 'day_of_week_number', 'day_of_week', 'month_number', 'month_name', 'quarter', 'time_period', 'resolved']


In [8]:
# ==========================================
# QUICK OVERVIEW
# ==========================================

# Ver primeras filas
print("📊 df_full - First 5 rows:")
print(df_full.head())

print("\n📊 df_match - First 5 rows:")
print(df_match.head())

📊 df_full - First 5 rows:
    interaction_id  hotel_id       channel language     request_type  \
0  INT2025_01_0001  hotel_01  website_chat       NO             info   
1  INT2025_01_0002  hotel_01         email       EN          booking   
2  INT2025_01_0003  hotel_03      facebook       NO  service_request   
3  INT2025_01_0004  hotel_07     instagram       EN        complaint   
4  INT2025_01_0005  hotel_02         email       NO         check_in   

  complexity               hotel_name       city country hotel_type  rooms  \
0     simple                 The Well   Sandvika  Norway        spa  231.0   
1    complex                 The Well   Sandvika  Norway        spa  231.0   
2     simple          Fjord Spa Hotel       Flåm  Norway        spa   95.0   
3    complex  Trondheim Central Hotel  Trondheim  Norway       city  140.0   
4     simple          Oslo City Hotel       Oslo  Norway       city  180.0   

  hotel_size_category         timestamp day_of_week  hour_of_day  \
0   

In [9]:
print("\nCities in df_full:")
print(df_full['city'].unique())


Cities in df_full:
<ArrowStringArray>
[    'Sandvika',         'Flåm',    'Trondheim',         'Oslo',
  'Lillehammer',         'Bodø',        'Geilo',    'Stavanger',
       'Bergen',      'Ålesund', 'Kristiansand',       'Tromsø',
            nan]
Length: 13, dtype: str


In [10]:
# ==========================================
# CHECK DATA TYPES
# ==========================================

print("df_full dtypes:")
print(df_full.dtypes)

df_full dtypes:
interaction_id             str
hotel_id                   str
channel                    str
language                   str
request_type               str
complexity                 str
hotel_name                 str
city                       str
country                    str
hotel_type                 str
rooms                  float64
hotel_size_category        str
timestamp                  str
day_of_week                str
hour_of_day              int64
day_of_week_number       int64
time_period                str
month_number             int64
month_name                 str
quarter                  int64
dtype: object


In [11]:
# ==========================================
# CHECK DATA TYPES
# ==========================================
print("\ndf_match dtypes:")
print(df_match.dtypes)


df_match dtypes:
interaction_id             str
hotel_id                   str
timestamp                  str
channel                    str
language                   str
request_type               str
complexity                 str
hotel_name                 str
city                       str
country                    str
hotel_type                 str
rooms                  float64
hotel_size_category        str
handled_by                 str
response_time_s        float64
response_speed             str
hour_of_day              int64
day_of_week_number       int64
day_of_week                str
month_number             int64
month_name                 str
quarter                  int64
time_period                str
resolved                   str
dtype: object


In [12]:
# ==========================================
# FIX TIMESTAMP - DD/MM/YYYY FORMAT
# ==========================================

# Fix timestamp con formato correcto
df_full['timestamp'] = pd.to_datetime(df_full['timestamp'], dayfirst=True)
df_match['timestamp'] = pd.to_datetime(df_match['timestamp'], dayfirst=True)



In [13]:
# Fix resolved en df_match
df_match['resolved'] = df_match['resolved'].map(
    {'True': True, 'False': False, 'TRUE': True, 'FALSE': False}
)

In [14]:
# Verificar
print("AFTER CONVERSION:")
print(f"df_full timestamp:  {df_full['timestamp'].dtype}")
print(f"df_match timestamp: {df_match['timestamp'].dtype}")
print(f"df_match resolved:  {df_match['resolved'].dtype}")
print(f"\nResolved values: {df_match['resolved'].value_counts()}")

AFTER CONVERSION:
df_full timestamp:  datetime64[us]
df_match timestamp: datetime64[us]
df_match resolved:  object

Resolved values: resolved
True     2925
False     361
Name: count, dtype: int64


In [15]:
# Ver valores únicos de resolved
print("Resolved unique values:")
print(df_match['resolved'].unique())

print("\nResolved value counts:")
print(df_match['resolved'].value_counts(dropna=False))

Resolved unique values:
[True False nan]

Resolved value counts:
resolved
True     2925
False     361
NaN         6
Name: count, dtype: int64


In [19]:
# ==========================================
# MISSING VALUES CHECK - DOCUMENTATION
# ==========================================

print("="*60)
print("MISSING VALUES VERIFICATION (from SQL cleaning)")
print("="*60)

print("\n📊 df_full (volume analysis):")
missing_full = pd.DataFrame({
    'missing_count': df_full.isnull().sum(),
    'missing_pct': (df_full.isnull().sum() / len(df_full) * 100).round(2)
})

missing_full_filtered = missing_full[missing_full['missing_count'] > 0]
print(missing_full_filtered)

missing_full_filtered.to_excel("missing_values.xlsx")  # Abre en Excel y copia
# O directo al portapapeles:
missing_full_filtered.to_clipboard()  # Ctrl+V en Word o Excel


MISSING VALUES VERIFICATION (from SQL cleaning)

📊 df_full (volume analysis):
                     missing_count  missing_pct
language                       888        18.73
request_type                    12         0.25
hotel_name                     438         9.24
city                           438         9.24
country                        438         9.24
hotel_type                     438         9.24
rooms                          438         9.24
hotel_size_category            438         9.24


In [20]:
print("\n📊 df_match (performance analysis):")
missing_match = pd.DataFrame({
    'missing_count': df_match.isnull().sum(),
    'missing_pct': (df_match.isnull().sum() / len(df_match) * 100).round(2)
})
print(missing_match[missing_match['missing_count'] > 0])

print("\n✅ Missing values confirmed - consistent with SQL validation")
print("="*60)


missing_match_filtered = missing_match[missing_match['missing_count'] > 0]
print(missing_full_filtered)

missing_match_filtered.to_excel("missing_values.xlsx")  # Abre en Excel y copia
# O directo al portapapeles:
missing_match_filtered.to_clipboard()  # Ctrl+V en Word o Excel


📊 df_match (performance analysis):
                     missing_count  missing_pct
language                       563        17.10
request_type                     8         0.24
hotel_name                     286         8.69
city                           286         8.69
country                        286         8.69
hotel_type                     286         8.69
rooms                          286         8.69
hotel_size_category            286         8.69
response_time_s                 25         0.76
response_speed                  25         0.76
resolved                         6         0.18

✅ Missing values confirmed - consistent with SQL validation
                     missing_count  missing_pct
language                       888        18.73
request_type                    12         0.25
hotel_name                     438         9.24
city                           438         9.24
country                        438         9.24
hotel_type                     438     

In [21]:
# ==========================================
# DUPLICATES CHECK - CONFIRMATION
# ==========================================

print("="*60)
print("DUPLICATES CHECK (from SQL cleaning)")
print("="*60)

# Check duplicates by interaction_id
print("\n📊 Duplicate interaction_ids:")
print(f"df_full:  {df_full['interaction_id'].duplicated().sum()} duplicates")
print(f"df_match: {df_match['interaction_id'].duplicated().sum()} duplicates")

# Verify uniqueness
print("\n📊 Uniqueness verification:")
print(f"df_full:  {len(df_full)} total rows, {df_full['interaction_id'].nunique()} unique IDs")
print(f"df_match: {len(df_match)} total rows, {df_match['interaction_id'].nunique()} unique IDs")

print("\n✅ No duplicates - SQL cleaning confirmed")
print("="*60)

DUPLICATES CHECK (from SQL cleaning)

📊 Duplicate interaction_ids:
df_full:  0 duplicates
df_match: 0 duplicates

📊 Uniqueness verification:
df_full:  4740 total rows, 4740 unique IDs
df_match: 3292 total rows, 3292 unique IDs

✅ No duplicates - SQL cleaning confirmed


In [22]:
# ==========================================
# BASIC STATISTICS - OVERVIEW
# ==========================================

print("="*60)
print("BASIC STATISTICS")
print("="*60)

print("\n📊 df_full - Numeric columns:")
print(df_full[['rooms', 'hour_of_day', 'day_of_week_number', 'month_number', 'quarter']].describe())

print("\n📊 df_match - Numeric columns (including handling):")
print(df_match[['rooms', 'hour_of_day', 'month_number', 'response_time_s']].describe())

print("\n📊 Response time statistics (key metric):")
print(f"Min:    {df_match['response_time_s'].min():.2f}s")
print(f"Max:    {df_match['response_time_s'].max():.2f}s")
print(f"Mean:   {df_match['response_time_s'].mean():.2f}s")
print(f"Median: {df_match['response_time_s'].median():.2f}s")

print("="*60)

BASIC STATISTICS

📊 df_full - Numeric columns:
             rooms  hour_of_day  day_of_week_number  month_number      quarter
count  4302.000000  4740.000000          4740.00000   4740.000000  4740.000000
mean    132.438168    11.961603             2.99346      6.687342     2.580380
std      36.311208     3.031209             2.00983      3.198393     1.047951
min      85.000000     3.000000             0.00000      1.000000     1.000000
25%     100.000000    10.000000             1.00000      4.000000     2.000000
50%     130.000000    12.000000             3.00000      7.000000     3.000000
75%     160.000000    15.000000             5.00000      9.000000     3.000000
max     231.000000    18.000000             6.00000     12.000000     4.000000

📊 df_match - Numeric columns (including handling):
             rooms  hour_of_day  month_number  response_time_s
count  3006.000000  3292.000000   3292.000000      3267.000000
mean    132.808383    11.938639      6.572904       175.156688
s

In [23]:
# ==========================================
# EXPORT VERIFICATION RESULTS TO EXCEL
# ==========================================

# Crear Excel con múltiples sheets
with pd.ExcelWriter('data_validation_report.xlsx', engine='openpyxl') as writer:
    
    # SHEET 1: Missing Values - df_full
    missing_full = pd.DataFrame({
        'column': df_full.columns,
        'missing_count': df_full.isnull().sum().values,
        'missing_pct': (df_full.isnull().sum() / len(df_full) * 100).round(2).values,
        'total_rows': len(df_full)
    })
    missing_full.to_excel(writer, sheet_name='Missing_df_full', index=False)
    
    # SHEET 2: Missing Values - df_match
    missing_match = pd.DataFrame({
        'column': df_match.columns,
        'missing_count': df_match.isnull().sum().values,
        'missing_pct': (df_match.isnull().sum() / len(df_match) * 100).round(2).values,
        'total_rows': len(df_match)
    })
    missing_match.to_excel(writer, sheet_name='Missing_df_match', index=False)
    
    # SHEET 3: Basic Statistics - df_full
    stats_full = df_full[['rooms', 'hour_of_day', 'day_of_week_number', 'month_number']].describe()
    stats_full.to_excel(writer, sheet_name='Stats_df_full')
    
    # SHEET 4: Basic Statistics - df_match
    stats_match = df_match[['rooms', 'hour_of_day', 'month_number', 'response_time_s']].describe()
    stats_match.to_excel(writer, sheet_name='Stats_df_match')
    
    # SHEET 5: Duplicates Check
    duplicates_check = pd.DataFrame({
        'dataset': ['df_full', 'df_match'],
        'total_rows': [len(df_full), len(df_match)],
        'unique_interaction_ids': [df_full['interaction_id'].nunique(), df_match['interaction_id'].nunique()],
        'duplicates': [df_full['interaction_id'].duplicated().sum(), df_match['interaction_id'].duplicated().sum()]
    })
    duplicates_check.to_excel(writer, sheet_name='Duplicates_Check', index=False)
    
    # SHEET 6: Overview
    overview = pd.DataFrame({
        'metric': [
            'df_full total rows',
            'df_full total columns',
            'df_match total rows',
            'df_match total columns',
            'Date range start',
            'Date range end'
        ],
        'value': [
            len(df_full),
            len(df_full.columns),
            len(df_match),
            len(df_match.columns),
            df_full['timestamp'].min(),
            df_full['timestamp'].max()
        ]
    })
    overview.to_excel(writer, sheet_name='Overview', index=False)

print("✅ Excel exported: data_validation_report.xlsx")
print("\nSheets created:")
print("  1. Missing_df_full")
print("  2. Missing_df_match")
print("  3. Stats_df_full")
print("  4. Stats_df_match")
print("  5. Duplicates_Check")
print("  6. Overview")

✅ Excel exported: data_validation_report.xlsx

Sheets created:
  1. Missing_df_full
  2. Missing_df_match
  3. Stats_df_full
  4. Stats_df_match
  5. Duplicates_Check
  6. Overview
